# LASS

In [ ]:
import numpy as np
import amplitf.interface as atfi
import amplitf.dynamics as atfd
import amplitf.kinematics as atfk
from amplitf.phasespace.dalitz_phasespace import DalitzPhaseSpace
import matplotlib.pyplot as plt
import tfa.plotting as tfp

In [ ]:
# LASS
lass_M = atfi.const(1.4617)
lass_G = atfi.const(0.2683)
lass_F = atfi.const(0.4524)
lass_phiF = atfi.const(0.248)
lass_R = atfi.const(1.0)
lass_phiR = atfi.const(2.495)
lass_a = atfi.const(0.172)
lass_r = atfi.const(-20.6)

In [ ]:
# Masses of final state particles
from particle.particle import literals as lp
# Dalitz Particles
mkz = atfi.const(lp.K_S_0.mass/1000)
mpi = atfi.const(lp.pi_plus.mass/1000)
md = atfi.const(lp.D_0.mass/1000)
# Dalitz Phase Space
phsp = DalitzPhaseSpace(mpi, mkz, mpi, md)

In [ ]:
def LASS(m2, a, r, m0, gamma0, md, mc, amp_res, amp_nr, phase_res, phase_nr):
    m = atfi.sqrt(m2)
    #q0 = atfk.two_body_momentum(md, m0, mc)
    q = atfk.two_body_momentum(md, m, mc)
    cot_delta_beta = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    #delta_beta = atfi.atan(1.0 / cot_delta_beta)
    nr = atfd.nonresonant_lass_lineshape(m2, a, r, md, mc)
    res= atfd.resonant_lass_lineshape(m2, m0, gamma0, a, r, md, mc)
    # lass = atfi.cast_complex(amp_res) * \
    #     atfi.complex( atfi.cos(phase_res + 2.0*phase_nr), atfi.sin(phase_res + 2.0*phase_nr) ) * \
    #          atfi.complex( q * cot_delta_beta, q ) / atfi.complex( q * cot_delta_beta, -q ) * res
    lass = atfi.cast_complex(amp_nr) * atfi.complex( atfi.cos(phase_nr), atfi.sin(phase_nr) ) * \
            atfi.cast_complex( atfi.cos(phase_nr) + atfi.sin(phase_nr) * cot_delta_beta ) #* nr
    return lass

In [ ]:
nnorm=100000
norm_sample = phsp.uniform_sample(nnorm)
m2ac = phsp.m2ac(norm_sample)
m2ab = phsp.m2ab(norm_sample)
m2bc = phsp.m2bc(norm_sample)
m_ac = np.sqrt(m2ac)
m_ab = np.sqrt(m2ab)
m_bc = np.sqrt(m2bc)

In [ ]:
lass_tf = LASS(phsp.m2ab(norm_sample), lass_a, lass_r, lass_M, lass_G, mkz, mpi, lass_R, lass_F, lass_phiR, lass_phiF)

In [ ]:
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_tf), bins=100, histtype='step', label='LASS Amplitf')
#plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(aBW_tf), bins=100, histtype='step', label='BW Amplitf', density=True)
plt.xlabel(r'$m_{ab}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ab')
plt.legend()

Let's now code Jordi's version of the LASS to check what is different.

In [ ]:
def lass_jordi(mab2, _amp, _phase, _LASS_a, _LASS_r, _massR, _gammaR, _ma, _mb, _LASS_R, _LASS_phi_R, _LASS_F, _LASS_phi_F):
    pi180inv = np.pi / 180.

    ampl=_amp
    theta=_phase

    s = mab2

    _a = _LASS_a
    _r = _LASS_r
    _R = _LASS_R # Amplitude magnitude of the resonant term
    _phiR = _LASS_phi_R # Phase of the resonant term
    _F = _LASS_F # Amplitude magnitude of the non-resonant term
    _phiF = _LASS_phi_F # Phase of the non-resonant term

    mR=_massR
    gammaR=_gammaR

    fR=1.0 # K*0(1430) has spin zero
    power=1 # Power is 1 for spin zero

    mAB=np.sqrt(mab2) # (_p4_d1+_p4_d2).mass();

    mA=_ma # _p4_d1.mass();
    mB=_mb # _p4_d2.mass();

    pAB=np.sqrt( (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB))
    q=pAB

    pR= np.sqrt( (((mR*mR-mA*mA-mB*mB)*(mR*mR-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mR*mR))

    # Running width.
    g = gammaR*np.power(pAB/pR,power)*(mR/mAB)*fR*fR

    propagator_relativistic_BreitWigner = 1./(mR*mR - mAB*mAB - 1j*mR*g)

	# Non-resonant phase shift
    cot_deltaF = 1.0/(_a*q) + 0.5*_r*q
    qcot_deltaF = 1.0/_a + 0.5*_r*q*q

	# Compute resonant part
    expi2deltaF = (qcot_deltaF + 1j*q)/ (qcot_deltaF - 1j*q)

    #resonant_term_T = _R * complex(np.cos(_phiR + 2 * _phiF), np.sin(_phiR + 2 * _phiF)) * propagator_relativistic_BreitWigner * mR * gammaR * mR / pR * expi2deltaF

	# Compute non-resonant part
    non_resonant_term_F = _F * complex(np.cos(_phiF), np.sin(_phiF)) * (np.cos(_phiF) + cot_deltaF * np.sin(_phiF)) #* np.sqrt(s) / (qcot_deltaF - 1j*q)

	# Add non-resonant and resonant terms
    LASS_contribution = non_resonant_term_F #+ resonant_term_T

    matrixEl = ampl * complex(np.cos(theta*pi180inv), np.sin(theta*pi180inv)) * LASS_contribution

    return matrixEl

In [ ]:
lass_jd = lass_jordi(phsp.m2ab(norm_sample), 1.0, 0.0, lass_a.numpy(), lass_r.numpy(), 
                    lass_M.numpy(), lass_G.numpy(), mkz.numpy(), mpi.numpy(), 
                    lass_R.numpy(), lass_phiR.numpy(), lass_F.numpy(), lass_phiF.numpy())

In [ ]:
plt.hist(phsp.m2ab(norm_sample), weights = np.power(np.abs(lass_jd),2), bins=100, histtype='step', label='LASS Jordi')
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_tf), bins=100, histtype='step', label='LASS Amplitf', density=True)
plt.xlabel(r'$m_{ab}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ab')
plt.legend()

In [ ]:
def nonresonant_lass_lineshape_jd(m2ab, a, r, ma, mb):
    r"""LASS line shape, nonresonant part

    .. math::

        LASS(m^2) = \frac{m}{q \cot \delta_b - i q}

    
    with :math:`q` is the momentum of the two-body system and :math:`\delta_b` is the scattering phase shift

    .. math::

        \cot \delta_b = \frac{1}{a q} + \frac{1}{2} r q


    from `Aston et al. Nuclear Physics B, Volume 296, Issue 3 (1988), Pages 493-526 <https://doi.org/10.1016/0550-3213(88)90028-4>`_

    Args:
        m2ab (float): invariant mass squared of the system
        a (float): parameter of the effective range term
        r (float): parameter of the effective range term
        ma (float): mass of particle a
        mb (float): mass of particle b

    Returns:
        complex: the nonresonant LASS amplitude
    """
    m = atfi.sqrt(m2ab)
    q = atfk.two_body_momentum(m, ma, mb)
    # q = atfk.two_body_momentum(m, ma, mb)
    cot_deltab = 1.0 / a / q + 1.0 / 2.0 * r * q
    ampl = atfi.cast_complex(m) / atfi.complex(q * cot_deltab, -q)
    return ampl

def resonant_lass_lineshape_jd(m2ab,
                            m0,
                            gamma0,
                            a,
                            r,
                            ma,
                            mb):
    r"""LASS line shape, resonant part

    .. math::

        LASS(m^2) = BW(m^2) (\cos \delta_b + i \sin \delta_b ) ( m_0^2 \Gamma_0 / q_0 )

    Args:
        m2ab (float): invariant mass squared of the system
        m0 (float): resonance mass
        gamma0 (float): resonance width
        a (float): parameter *a* of the effective range term
        r (float): parameter *r* of the effective range term
        md (float): mass of mother particle 
        mc (float): mass of other particle wrt resonance

    Returns:
        complex: the resonant LASS amplitude
    """
    m = atfi.sqrt(m2ab)
    q = atfk.two_body_momentum(m, ma, mb)
    q0 = atfk.two_body_momentum(m0, ma, mb)
    cot_deltab = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    phase = atfi.atan(1.0 / cot_deltab)
    width = gamma0 * q / m * m0 / q0
    ampl = (atfd.relativistic_breit_wigner(m2ab, m0, width) *
            atfi.complex(atfi.cos(phase), atfi.sin(phase)) *
            atfi.cast_complex(m0 * m0 * gamma0 / q0))
    return ampl

def LASS_jordi_tf(m2, a, r, m0, gamma0, ma, mb, amp_res, amp_nr, phase_res, phase_nr):
    m = atfi.sqrt(m2)
    #q0 = atfk.two_body_momentum(md, m0, mc)
    q = atfk.two_body_momentum(m, ma, mb)
    cot_delta_beta = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    #delta_beta = atfi.atan(1.0 / cot_delta_beta)
    nr = nonresonant_lass_lineshape_jd(m2, a, r, ma, mb)
    res= resonant_lass_lineshape_jd(m2, m0, gamma0, a, r, ma, mb)
    lass = atfi.cast_complex(amp_res) * \
        atfi.complex( atfi.cos(phase_res + 2.0*phase_nr), atfi.sin(phase_res + 2.0*phase_nr) ) * \
             atfi.complex( q * cot_delta_beta, q ) / atfi.complex( q * cot_delta_beta, -q ) * res
    lass += atfi.cast_complex(amp_nr) * atfi.complex( atfi.cos(phase_nr), atfi.sin(phase_nr) ) * \
           atfi.cast_complex( atfi.cos(phase_nr) + atfi.sin(phase_nr) * cot_delta_beta ) * nr
    return lass

def lass_jordi(mab2, _amp, _phase, _LASS_a, _LASS_r, _massR, _gammaR, _ma, _mb, _LASS_R, _LASS_phi_R, _LASS_F, _LASS_phi_F):
    pi180inv = np.pi / 180.

    ampl=_amp
    theta=_phase

    s = mab2

    _a = _LASS_a
    _r = _LASS_r
    _R = _LASS_R # Amplitude magnitude of the resonant term
    _phiR = _LASS_phi_R # Phase of the resonant term
    _F = _LASS_F # Amplitude magnitude of the non-resonant term
    _phiF = _LASS_phi_F # Phase of the non-resonant term

    mR=_massR
    gammaR=_gammaR

    fR=1.0 # K*0(1430) has spin zero
    power=1 # Power is 1 for spin zero

    mAB=np.sqrt(mab2) # (_p4_d1+_p4_d2).mass();

    mA=_ma # _p4_d1.mass();
    mB=_mb # _p4_d2.mass();

    pAB=np.sqrt( (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB))
    q=pAB

    pR= np.sqrt( (((mR*mR-mA*mA-mB*mB)*(mR*mR-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mR*mR))

    # Running width.
    g = gammaR*np.power(pAB/pR,power)*(mR/mAB)*fR*fR

    propagator_relativistic_BreitWigner = 1./(mR*mR - mAB*mAB - 1j*mR*g)

	# Non-resonant phase shift
    cot_deltaF = 1.0/(_a*q) + 0.5*_r*q
    qcot_deltaF = 1.0/_a + 0.5*_r*q*q

	# Compute resonant part
    expi2deltaF = (qcot_deltaF + 1j*q)/ (qcot_deltaF - 1j*q)

    resonant_term_T = _R * complex(np.cos(_phiR + 2 * _phiF), np.sin(_phiR + 2 * _phiF)) * expi2deltaF * propagator_relativistic_BreitWigner * mR * gammaR * mR / pR 

	# Compute non-resonant part
    non_resonant_term_F = _F * complex(np.cos(_phiF), np.sin(_phiF)) * (np.cos(_phiF) + cot_deltaF * np.sin(_phiF)) * np.sqrt(s) / (qcot_deltaF - 1j*q)

	# Add non-resonant and resonant terms
    LASS_contribution = non_resonant_term_F + resonant_term_T
    #LASS_contribution = resonant_term_T

    matrixEl = ampl * complex(np.cos(theta*pi180inv), np.sin(theta*pi180inv)) * LASS_contribution

    return matrixEl

lass_jd = lass_jordi(phsp.m2ab(norm_sample), 1.0, 0.0, lass_a.numpy(), lass_r.numpy(), 
                    lass_M.numpy(), lass_G.numpy(), mkz.numpy(), mpi.numpy(), 
                    lass_R.numpy(), lass_phiR.numpy(), lass_F.numpy(), lass_phiF.numpy())


lass_tf = LASS_jordi_tf(phsp.m2ab(norm_sample), lass_a, lass_r, lass_M, lass_G, mkz, mpi, lass_R, lass_F, lass_phiR, lass_phiF)
plt.hist(phsp.m2ab(norm_sample), weights = np.power(np.abs(lass_jd),2), bins=100, histtype='step', label='LASS Jordi', density=True)
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_tf), bins=100, histtype='step', label='LASS Amplitf', density=True)
plt.xlabel(r'$m_{ab}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ab')
plt.legend()

Plot resonant and non-resonant components

In [ ]:
def lass_jordi_nr(mab2, _amp, _phase, _LASS_a, _LASS_r, _massR, _gammaR, _ma, _mb, _LASS_R, _LASS_phi_R, _LASS_F, _LASS_phi_F):
    pi180inv = np.pi / 180.

    ampl=_amp
    theta=_phase

    s = mab2

    _a = _LASS_a
    _r = _LASS_r
    _R = _LASS_R # Amplitude magnitude of the resonant term
    _phiR = _LASS_phi_R # Phase of the resonant term
    _F = _LASS_F # Amplitude magnitude of the non-resonant term
    _phiF = _LASS_phi_F # Phase of the non-resonant term

    mR=_massR
    gammaR=_gammaR

    fR=1.0 # K*0(1430) has spin zero
    power=1 # Power is 1 for spin zero

    mAB=np.sqrt(mab2) # (_p4_d1+_p4_d2).mass();

    mA=_ma # _p4_d1.mass();
    mB=_mb # _p4_d2.mass();

    pAB=np.sqrt( (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB))
    q=pAB

    pR= np.sqrt( (((mR*mR-mA*mA-mB*mB)*(mR*mR-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mR*mR))

    # Running width.
    g = gammaR*np.power(pAB/pR,power)*(mR/mAB)*fR*fR

    propagator_relativistic_BreitWigner = 1./(mR*mR - mAB*mAB - 1j*mR*g)

	# Non-resonant phase shift
    cot_deltaF = 1.0/(_a*q) + 0.5*_r*q
    qcot_deltaF = 1.0/_a + 0.5*_r*q*q

	# Compute resonant part
    expi2deltaF = (qcot_deltaF + 1j*q)/ (qcot_deltaF - 1j*q)

    #resonant_term_T = _R * complex(np.cos(_phiR + 2 * _phiF), np.sin(_phiR + 2 * _phiF)) * expi2deltaF * propagator_relativistic_BreitWigner * mR * gammaR * mR / pR 

	# Compute non-resonant part
    non_resonant_term_F = _F * complex(np.cos(_phiF), np.sin(_phiF)) * (np.cos(_phiF) + cot_deltaF * np.sin(_phiF)) * np.sqrt(s) / (qcot_deltaF - 1j*q)

	# Add non-resonant and resonant terms
    LASS_contribution = non_resonant_term_F
    #LASS_contribution = resonant_term_T

    #matrixEl = ampl * complex(np.cos(theta*pi180inv), np.sin(theta*pi180inv)) * LASS_contribution
    matrixEl = LASS_contribution

    return matrixEl


def lass_jordi_res(mab2, _amp, _phase, _LASS_a, _LASS_r, _massR, _gammaR, _ma, _mb, _LASS_R, _LASS_phi_R, _LASS_F, _LASS_phi_F):
    pi180inv = np.pi / 180.

    ampl=_amp
    theta=_phase

    s = mab2

    _a = _LASS_a
    _r = _LASS_r
    _R = _LASS_R # Amplitude magnitude of the resonant term
    _phiR = _LASS_phi_R # Phase of the resonant term
    _F = _LASS_F # Amplitude magnitude of the non-resonant term
    _phiF = _LASS_phi_F # Phase of the non-resonant term

    mR=_massR
    gammaR=_gammaR

    fR=1.0 # K*0(1430) has spin zero
    power=1 # Power is 1 for spin zero

    mAB=np.sqrt(mab2) # (_p4_d1+_p4_d2).mass();

    mA=_ma # _p4_d1.mass();
    mB=_mb # _p4_d2.mass();

    pAB=np.sqrt( (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB))
    q=pAB

    pR= np.sqrt( (((mR*mR-mA*mA-mB*mB)*(mR*mR-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mR*mR))

    # Running width.
    g = gammaR*np.power(pAB/pR,power)*(mR/mAB)*fR*fR

    propagator_relativistic_BreitWigner = 1./(mR*mR - mAB*mAB - 1j*mR*g)

	# Non-resonant phase shift
    cot_deltaF = 1.0/(_a*q) + 0.5*_r*q
    qcot_deltaF = 1.0/_a + 0.5*_r*q*q

	# Compute resonant part
    expi2deltaF = (qcot_deltaF + 1j*q)/ (qcot_deltaF - 1j*q)

    resonant_term_T = _R * complex(np.cos(_phiR + 2 * _phiF), np.sin(_phiR + 2 * _phiF)) * expi2deltaF * propagator_relativistic_BreitWigner * mR * gammaR * mR / pR 

	# Compute non-resonant part
    #non_resonant_term_F = _F * complex(np.cos(_phiF), np.sin(_phiF)) * (np.cos(_phiF) + cot_deltaF * np.sin(_phiF)) * np.sqrt(s) / (qcot_deltaF - 1j*q)

	# Add non-resonant and resonant terms
    #LASS_contribution = non_resonant_term_F + resonant_term_T
    LASS_contribution = resonant_term_T

    #matrixEl = ampl * complex(np.cos(theta*pi180inv), np.sin(theta*pi180inv)) * LASS_contribution
    matrixEl = LASS_contribution

    return matrixEl

def LASS_jordi_nr_tf(m2, a, r, m0, gamma0, ma, mb, amp_res, amp_nr, phase_res, phase_nr):
    m = atfi.sqrt(m2)
    #q0 = atfk.two_body_momentum(md, m0, mc)
    q = atfk.two_body_momentum(m, ma, mb)
    cot_delta_beta = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    #delta_beta = atfi.atan(1.0 / cot_delta_beta)
    nr = nonresonant_lass_lineshape_jd(m2, a, r, ma, mb)
    res= resonant_lass_lineshape_jd(m2, m0, gamma0, a, r, ma, mb)
    # lass = atfi.cast_complex(amp_res) * \
    #     atfi.complex( atfi.cos(phase_res + 2.0*phase_nr), atfi.sin(phase_res + 2.0*phase_nr) ) * \
    #          atfi.complex( q * cot_delta_beta, q ) / atfi.complex( q * cot_delta_beta, -q ) * res
    lass = atfi.cast_complex(amp_nr) * atfi.complex( atfi.cos(phase_nr), atfi.sin(phase_nr) ) * \
           atfi.cast_complex( atfi.cos(phase_nr) + atfi.sin(phase_nr) * cot_delta_beta ) * nr
    return lass

def LASS_jordi_res_tf(m2, a, r, m0, gamma0, ma, mb, amp_res, amp_nr, phase_res, phase_nr):
    m = atfi.sqrt(m2)
    #q0 = atfk.two_body_momentum(md, m0, mc)
    q = atfk.two_body_momentum(m, ma, mb)
    cot_delta_beta = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    #delta_beta = atfi.atan(1.0 / cot_delta_beta)
    nr = nonresonant_lass_lineshape_jd(m2, a, r, ma, mb)
    res= resonant_lass_lineshape_jd(m2, m0, gamma0, a, r, ma, mb)
    lass = atfi.cast_complex(amp_res) * \
        atfi.complex( atfi.cos(phase_res + 2.0*phase_nr), atfi.sin(phase_res + 2.0*phase_nr) ) * \
             atfi.complex( q * cot_delta_beta, q ) / atfi.complex( q * cot_delta_beta, -q ) * res
    # lass = atfi.cast_complex(amp_nr) * atfi.complex( atfi.cos(phase_nr), atfi.sin(phase_nr) ) * \
    #        atfi.cast_complex( atfi.cos(phase_nr) + atfi.sin(phase_nr) * cot_delta_beta ) * nr
    return lass

In [ ]:
lass_jd_nr = lass_jordi_nr(phsp.m2ab(norm_sample), 1.0, 0.0, lass_a.numpy(), lass_r.numpy(), 
                    lass_M.numpy(), lass_G.numpy(), mkz.numpy(), mpi.numpy(), 
                    lass_R.numpy(), lass_phiR.numpy(), lass_F.numpy(), lass_phiF.numpy())
lass_nr_tf = LASS_jordi_nr_tf(phsp.m2ab(norm_sample), lass_a, lass_r, lass_M, lass_G, mkz, mpi, lass_R, lass_F, lass_phiR, lass_phiF)
lass_jd_res = lass_jordi_res(phsp.m2ab(norm_sample), 1.0, 0.0, lass_a.numpy(), lass_r.numpy(), 
                    lass_M.numpy(), lass_G.numpy(), mkz.numpy(), mpi.numpy(), 
                    lass_R.numpy(), lass_phiR.numpy(), lass_F.numpy(), lass_phiF.numpy())
lass_res_tf = LASS_jordi_res_tf(phsp.m2ab(norm_sample), lass_a, lass_r, lass_M, lass_G, mkz, mpi, lass_R, lass_F, lass_phiR, lass_phiF)
plt.hist(phsp.m2ab(norm_sample), weights = np.power(np.abs(lass_jd_nr),2), bins=100, histtype='step', label='LASS Jordi (NR)', density=True)
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_nr_tf), bins=100, histtype='step', label='LASS Amplitf (NR)', density=True)
plt.hist(phsp.m2ab(norm_sample), weights = np.power(np.abs(lass_jd_res),2), bins=100, histtype='step', label='LASS Jordi (Res)', density=True)
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_res_tf), bins=100, histtype='step', label='LASS Amplitf (Res)', density=True)
plt.xlabel(r'$m_{ab}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ab')
plt.legend()

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.real(lass_jd_nr), np.imag(lass_jd_nr), 'o')
ax.plot(np.real(lass_nr_tf), np.imag(lass_nr_tf), 'o')
ax.plot(np.real(lass_jd_res), np.imag(lass_jd_res), 'o')
ax.plot(np.real(lass_res_tf), np.imag(lass_res_tf), 'o')

The amplitudes are separately the same. Try to combine them again

In [ ]:
def lass_jordi_bis(mab2, _amp, _phase, _LASS_a, _LASS_r, _massR, _gammaR, _ma, _mb, _LASS_R, _LASS_phi_R, _LASS_F, _LASS_phi_F):
    pi180inv = np.pi / 180.

    ampl=_amp
    theta=_phase

    s = mab2

    _a = _LASS_a
    _r = _LASS_r
    _R = _LASS_R # Amplitude magnitude of the resonant term
    _phiR = _LASS_phi_R # Phase of the resonant term
    _F = _LASS_F # Amplitude magnitude of the non-resonant term
    _phiF = _LASS_phi_F # Phase of the non-resonant term

    mR=_massR
    gammaR=_gammaR

    fR=1.0 # K*0(1430) has spin zero
    power=1 # Power is 1 for spin zero

    mAB=np.sqrt(mab2) # (_p4_d1+_p4_d2).mass();

    mA=_ma # _p4_d1.mass();
    mB=_mb # _p4_d2.mass();

    pAB=np.sqrt( (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB))
    q=pAB

    pR= np.sqrt( (((mR*mR-mA*mA-mB*mB)*(mR*mR-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mR*mR))

    # Running width.
    g = gammaR*np.power(pAB/pR,power)*(mR/mAB)*fR*fR

    propagator_relativistic_BreitWigner = 1./(mR*mR - mAB*mAB - 1j*mR*g)

	# Non-resonant phase shift
    cot_deltaF = 1.0/(_a*q) + 0.5*_r*q
    qcot_deltaF = 1.0/_a + 0.5*_r*q*q

	# Compute resonant part
    expi2deltaF = (qcot_deltaF + 1j*q)/ (qcot_deltaF - 1j*q)

    resonant_term_T = _R * complex(np.cos(_phiR + 2 * _phiF), np.sin(_phiR + 2 * _phiF)) * expi2deltaF * propagator_relativistic_BreitWigner * mR * gammaR * mR / pR 

	# Compute non-resonant part
    non_resonant_term_F = _F * complex(np.cos(_phiF), np.sin(_phiF)) * (np.cos(_phiF) + cot_deltaF * np.sin(_phiF)) * np.sqrt(s) / (qcot_deltaF - 1j*q)

	# Add non-resonant and resonant terms
    LASS_contribution = non_resonant_term_F + resonant_term_T

    #matrixEl = ampl * complex(np.cos(theta*pi180inv), np.sin(theta*pi180inv)) * LASS_contribution
    matrixEl = LASS_contribution

    return matrixEl

def resonant_lass_lineshape_jd_bis(m2ab,
                            m0,
                            gamma0,
                            a,
                            r,
                            ma,
                            mb):
    r"""LASS line shape, resonant part

    .. math::

        LASS(m^2) = BW(m^2) (\cos \delta_b + i \sin \delta_b ) ( m_0^2 \Gamma_0 / q_0 )

    Args:
        m2ab (float): invariant mass squared of the system
        m0 (float): resonance mass
        gamma0 (float): resonance width
        a (float): parameter *a* of the effective range term
        r (float): parameter *r* of the effective range term
        md (float): mass of mother particle 
        mc (float): mass of other particle wrt resonance

    Returns:
        complex: the resonant LASS amplitude
    """
    m = atfi.sqrt(m2ab)
    q = atfk.two_body_momentum(m, ma, mb)
    q0 = atfk.two_body_momentum(m0, ma, mb)
    cot_deltab = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    phase = atfi.atan(1.0 / cot_deltab)
    width = gamma0 * q / m * m0 / q0
    ampl = (atfd.relativistic_breit_wigner(m2ab, m0, width) *
            #atfi.complex(atfi.cos(phase), atfi.sin(phase)) *
            atfi.cast_complex(m0 * m0 * gamma0 / q0))
    return ampl

def LASS_jordi_tf_bis(m2, a, r, m0, gamma0, ma, mb, amp_res, phase_res, amp_nr, phase_nr):
    m = atfi.sqrt(m2)
    #q0 = atfk.two_body_momentum(md, m0, mc)
    q = atfk.two_body_momentum(m, ma, mb)
    cot_delta_beta = atfi.const(1.0) / a / q + atfi.const(0.5) * r * q
    #delta_beta = atfi.atan(1.0 / cot_delta_beta)
    nr = nonresonant_lass_lineshape_jd(m2, a, r, ma, mb)
    res= resonant_lass_lineshape_jd_bis(m2, m0, gamma0, a, r, ma, mb)
    lass = atfi.cast_complex(amp_res) * \
        atfi.complex( atfi.cos(phase_res + 2.0*phase_nr), atfi.sin(phase_res + 2.0*phase_nr) ) * \
             atfi.complex( q * cot_delta_beta, q ) / atfi.complex( q * cot_delta_beta, -q ) * res
    lass += atfi.cast_complex(amp_nr) * atfi.complex( atfi.cos(phase_nr), atfi.sin(phase_nr) ) * \
           atfi.cast_complex( atfi.cos(phase_nr) + atfi.sin(phase_nr) * cot_delta_beta ) * nr
    return lass


lass_jd_bis = lass_jordi_bis(phsp.m2ab(norm_sample), 1.0, 0.0, lass_a.numpy(), lass_r.numpy(), 
                    lass_M.numpy(), lass_G.numpy(), mkz.numpy(), mpi.numpy(), 
                    lass_R.numpy(), lass_phiR.numpy(), lass_F.numpy(), lass_phiF.numpy())
lass_bis_tf = LASS_jordi_tf_bis(phsp.m2ab(norm_sample), lass_a, lass_r, lass_M, lass_G, mkz, mpi, lass_R, lass_phiR, lass_F, lass_phiF)
plt.hist(phsp.m2ab(norm_sample), weights = np.power(np.abs(lass_jd_bis),2), bins=100, histtype='step', label='LASS Jordi (Sum)', density=True)
plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(lass_bis_tf), bins=100, histtype='step', label='LASS Amplitf (Sum)', density=True)
plt.xlabel(r'$m_{ab}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ab')
plt.legend()

# LASS: comparison with the C++ code

The C++ implementation (`Resonance.cpp` in d02kspipi_toys) is no longer loaded with ROOT here. Its values at 2000 Dalitz points are stored in `inputs/d02kspipi_cpp_reference.npz` (produced by `macros/make_cpp_reference.py`), and `tests/test_cpp_reference.py` checks every term of the model against them. The comparison uses the Belle 2018 parameters and the masses of the C++ code. The previous ROOT-based cells are in the git history of `main`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tfaex.reference import load_cpp_reference, tf_components
from tfaex.plotting import plot_reference_comparison

ref = load_cpp_reference()
tf = tf_components(ref)
m_pipi, m_kspim, m_kspip = (np.sqrt(ref[k]) for k in ("m2_pipi", "m2_kspim", "m2_kspip"))

In [ ]:
plot_reference_comparison(m_kspim, ref["Kstarzero1430minus"], tf["Kstarzero1430minus"], r"$m(K_S^0\pi^-)$ [GeV]", title="Kstarzero1430minus")
plot_reference_comparison(m_kspip, ref["Kstarzero1430plus"], tf["Kstarzero1430plus"], r"$m(K_S^0\pi^+)$ [GeV]", title="Kstarzero1430plus");

The two agree to machine precision once the fitted $K^*_0(1430)$ mass and width (`Kstarzero1430_Mass`, `Kstarzero1430_Width` in `params/belle_model.txt`) are used, as `tfaex.models.belle2018` now does.